# Let's try some **regression**!!

## Simple Regression

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import src.machine_learning as ML

class GlyphClassifier(nn.Module):
    def __init__(self, resolution):
        super(GlyphClassifier, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * resolution[0]//8 * resolution[1]//8, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten the output
        x = self.classifier(x)
        return x

In [ ]:
# Getting the dataset

dataset_file = 'data/simple-star.zip'
train_dataset = ML.GlyphDataset(dataset_file, resize=(128,128), split = "train")
test_dataset = ML.GlyphDataset(dataset_file, resize=(128,128),split = 'test')

# Assign the loaders 

train_loader = ML.create_loader(train_dataset, batch_size=64, shuffle = True)
test_loader = ML.create_loader(test_dataset, batch_size=64, shuffle = False)
ML.visualize_loader(train_loader,max_images=10,nrow=5)


In [ ]:
import torch
import matplotlib.pyplot as plt
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = GlyphClassifier(resolution=(128, 128)).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# training parameters
num_epochs = 50
losses = []
steps = 0 

# Initialising Weights&Biasis
wandb.init(
    project="glyph-regression",
    name="exp-1024neuron-128x128res-simplestar-logging steps",
    config={
        "architecture": "CNN-Glyph",
        "epochs": num_epochs,
        "batch_size": 64,
        "learning_rate": 0.0005,
        "loss_fn": "SmoothL1Loss",
        "optimizer": "Adam",
        "image_resolution": (128, 128),
        "regression": True
    }
)

wandb.watch(model, log="all", log_freq=10)

for epoch in range(num_epochs):
    model.train()
    for images, values in train_loader:
        labels = torch.tensor(values, dtype=torch.float32).unsqueeze(1).to(device) 
        images = images.to(device)
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels.squeeze())
        loss.backward()
        optimizer.step()        
        losses.append(loss.item())
        steps += 1  
        wandb.log({
            "step": steps,
            "train_loss": loss.item(),
            "epoch": epoch + 1 
        })      
        print(f"Step {steps} - Epoch {epoch+1}/{num_epochs} - Loss: {loss.item():.4f}")

ML.plot_training_loss(losses)

In [ ]:
import numpy as np 

model.eval()
predictions = []
ground_truths = []
with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        outputs = model(images).squeeze()
        predictions.extend(outputs.cpu().numpy())
        ground_truths.extend(labels.cpu().numpy())
predictions = np.array(predictions)
ground_truths = np.array(ground_truths)
mse = np.mean((predictions - ground_truths)**2)
mae = np.mean(np.abs(predictions - ground_truths))
print(f"Regression MSE: {mse:.4f}")
print(f"Regression MAE: {mae:.4f}")
wandb.log({
    "test_mse": mse,
    "test_mae": mae
})

wandb.finish()